# Frugal AI: Model Comparison & Carbon Footprint

**Theme:** Frugal AI

This notebook:
1. Compares 3 models: Logistic Regression vs Random Forest vs XGBoost
2. Measures training time and CO₂ emissions with **CodeCarbon**
3. Makes the case for choosing the simplest model that is good enough
4. Visualizes the accuracy vs cost trade-off

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import os
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
    print("XGBoost available ✓")
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost not installed — will use GradientBoosting as fallback.")
    from sklearn.ensemble import GradientBoostingClassifier

try:
    from codecarbon import EmissionsTracker
    CODECARBON_AVAILABLE = True
    print("CodeCarbon available ✓")
except ImportError:
    CODECARBON_AVAILABLE = False
    print("CodeCarbon not installed — using time proxy instead.")
    print("Install with: pip install codecarbon")

# Ensure output directory exists
os.makedirs("../docs", exist_ok=True)

print("Libraries loaded ✓")

XGBoost available ✓
CodeCarbon available ✓
Libraries loaded ✓


## 1. Load Data

In [2]:
df = pd.read_csv('../data/processed/hr_anonymized.csv')

TARGET = 'Termd'
DROP_COLS = [c for c in ['Termd', 'TermReason', 'EmploymentStatus',
                          'DateofTermination', 'LastPerformanceReview_Date',
                          'PositionID', 'Position'] if c in df.columns]

X = df.drop(columns=DROP_COLS).select_dtypes(include=[np.number])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Features: {X.shape[1]} | Train: {len(X_train)} | Test: {len(X_test)}')

Features: 20 | Train: 248 | Test: 63


## 2. Define Models

> **Frugal principle**: start simple. Only add complexity if the performance gain justifies the cost.

In [3]:
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(
            max_iter=500,
            class_weight='balanced',
            random_state=42
        ))
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        max_depth=6,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
        random_state=42,
        eval_metric='logloss',
        verbosity=0
    )
}

print('Models defined:')
for name in models:
    print(f'  - {name}')

Models defined:
  - Logistic Regression
  - Random Forest
  - XGBoost


## 3. Train & Measure — Accuracy + Carbon Footprint

In [5]:
results = []

for name, model in models.items():
    print(f'\nTraining: {name}...')

    if CODECARBON_AVAILABLE:
        tracker = EmissionsTracker(
            project_name=f'hackathon_{name.replace(" ", "_").lower()}',
            output_dir='../docs/',
            log_level='error'
        )
        tracker.start()

    t0 = time.perf_counter()

    # 5-fold cross-validation
    cv_roc = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc', n_jobs=-1)
    cv_f1  = cross_val_score(model, X_train, y_train, cv=5, scoring='f1', n_jobs=-1)

    # Final fit on full train set
    model.fit(X_train, y_train)

    elapsed = time.perf_counter() - t0

    if CODECARBON_AVAILABLE:
        emissions = tracker.stop()  # kg CO2eq
        emissions_g = emissions * 1000  # convert to grams
    else:
        emissions_g = None

    results.append({
        'Model': name,
        'ROC-AUC (CV mean)': cv_roc.mean(),
        'ROC-AUC (CV std)':  cv_roc.std(),
        'F1 (CV mean)':      cv_f1.mean(),
        'Training time (s)': elapsed,
        'CO2 (g)':           emissions_g
    })

    print(f'  ROC-AUC: {cv_roc.mean():.3f} ± {cv_roc.std():.3f}')
    print(f'  F1:      {cv_f1.mean():.3f}')
    print(f'  Time:    {elapsed:.2f}s')
    if emissions_g is not None:
        print(f'  CO2:     {emissions_g:.4f}g')

results_df = pd.DataFrame(results)
print('\nFull Comparison')
print(results_df.to_string(index=False))


Training: Logistic Regression...
  ROC-AUC: 1.000 ± 0.000
  F1:      0.988
  Time:    0.04s
  CO2:     0.0000g

Training: Random Forest...
  ROC-AUC: 1.000 ± 0.000
  F1:      0.994
  Time:    0.39s
  CO2:     0.0003g

Training: XGBoost...
  ROC-AUC: 0.994 ± 0.012
  F1:      0.988
  Time:    0.21s
  CO2:     0.0002g

Full Comparison
              Model  ROC-AUC (CV mean)  ROC-AUC (CV std)  F1 (CV mean)  Training time (s)  CO2 (g)
Logistic Regression           1.000000          0.000000      0.987879           0.042388 0.000032
      Random Forest           1.000000          0.000000      0.993939           0.386089 0.000292
            XGBoost           0.994118          0.011765      0.988225           0.206688 0.000156
